In [1]:
from google.colab import files
uploaded = files.upload()

Saving Messy_data.csv to Messy_data.csv


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("Messy_data.csv")

print(df.head())
print(df.info())
print(df.shape)
print(df.isnull().sum())

   OrderID  CustomerName     City     Product Quantity UnitPrice   OrderDate  \
0    10001   Nikhil Jain   Mumbai      Laptop      3.0  50790.72  2026-01-18   
1    10002   Simran Kaur   delhi      Monitor      1.0  10307.27  2026-01-28   
2    10003      Zoya Ali    Delhi  Headphones      5.0   2439.64  2026-02-27   
3    10004  Priya Bansal    Delhi      Tablet      4.0  20945.65  02/03/2026   
4    10005           NaN   Mumbai       Phone      4.0  15822.27  2026-02-14   

  OrderStatus CustomerAge  
0   Completed        61.0  
1     Pending        50.0  
2   Delivered        35.0  
3     Pending        66.0  
4   Delivered        34.0  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 185 entries, 0 to 184
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   OrderID       185 non-null    int64 
 1   CustomerName  183 non-null    object
 2   City          183 non-null    object
 3   Product       185 non-null    ob

In [3]:
# Remove duplicate records
df = df.drop_duplicates()

# Remove extra spaces from text columns
text_cols = ["CustomerName", "City", "Product", "OrderStatus"]

for col in text_cols:
    df[col] = df[col].astype("string").str.strip()

# Standardize text
df["City"] = df["City"].str.title()
df["Product"] = df["Product"].str.title()
df["OrderStatus"] = df["OrderStatus"].str.title()

# Handle missing names, cities, products and statuses
df["CustomerName"] = df["CustomerName"].fillna("Unknown")
df["City"] = df["City"].fillna("Unknown")
df["Product"] = df["Product"].fillna("Unknown")
df["OrderStatus"] = df["OrderStatus"].fillna("Unknown")

# Clean Quantity
df["Quantity"] = df["Quantity"].astype(str).str.strip()

df["Quantity"] = df["Quantity"].replace({
    "three": "3",
    "2 pcs": "2"
})

df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")

# Handle invalid and unusually large quantities
df.loc[(df["Quantity"] <= 0) | (df["Quantity"] > 50), "Quantity"] = np.nan

df["Quantity"] = df["Quantity"].fillna(df["Quantity"].median())
df["Quantity"] = df["Quantity"].round().astype(int)

# Clean UnitPrice
df["UnitPrice"] = df["UnitPrice"].astype(str).str.replace(",", "", regex=False)

df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")

# Handle extremely high prices
df.loc[df["UnitPrice"] > 100000, "UnitPrice"] = np.nan

df["UnitPrice"] = df["UnitPrice"].fillna(df["UnitPrice"].median())

# Clean CustomerAge
df["CustomerAge"] = df["CustomerAge"].astype(str).str.strip()

df["CustomerAge"] = df["CustomerAge"].replace({
    "forty": "40",
    "unknown": np.nan
})

df["CustomerAge"] = pd.to_numeric(df["CustomerAge"], errors="coerce")

# Handle unrealistic ages
df.loc[(df["CustomerAge"] < 18) | (df["CustomerAge"] > 100), "CustomerAge"] = np.nan

df["CustomerAge"] = df["CustomerAge"].fillna(df["CustomerAge"].median())
df["CustomerAge"] = df["CustomerAge"].round().astype(int)

# Clean dates with different formats
df["OrderDate"] = pd.to_datetime(
    df["OrderDate"],
    format="mixed",
    errors="coerce",
    dayfirst=True
)

# Remove records with invalid or missing dates
df = df.dropna(subset=["OrderDate"])

# Create TotalAmount
df["TotalAmount"] = df["Quantity"] * df["UnitPrice"]

print("Data cleaning completed!")
print(df.head())

Data cleaning completed!
   OrderID  CustomerName    City     Product  Quantity  UnitPrice  OrderDate  \
0    10001   Nikhil Jain  Mumbai      Laptop         3   50790.72 2026-01-18   
1    10002   Simran Kaur   Delhi     Monitor         1   10307.27 2026-01-28   
2    10003      Zoya Ali   Delhi  Headphones         5    2439.64 2026-02-27   
3    10004  Priya Bansal   Delhi      Tablet         4   20945.65 2026-03-02   
4    10005       Unknown  Mumbai       Phone         4   15822.27 2026-02-14   

  OrderStatus  CustomerAge  TotalAmount  
0   Completed           61    152372.16  
1     Pending           50     10307.27  
2   Delivered           35     12198.20  
3     Pending           66     83782.60  
4   Delivered           34     63289.08  


In [4]:
print("Final Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nData Types:")
print(df.dtypes)

print("\nDuplicate Records:")
print(df.duplicated().sum())

print("\nTotal Amount Summary:")
print(df["TotalAmount"].describe())

Final Shape: (177, 10)

Missing Values:
OrderID         0
CustomerName    0
City            0
Product         0
Quantity        0
UnitPrice       0
OrderDate       0
OrderStatus     0
CustomerAge     0
TotalAmount     0
dtype: int64

Data Types:
OrderID                  int64
CustomerName    string[python]
City            string[python]
Product         string[python]
Quantity                 int64
UnitPrice              float64
OrderDate       datetime64[ns]
OrderStatus     string[python]
CustomerAge              int64
TotalAmount            float64
dtype: object

Duplicate Records:
0

Total Amount Summary:
count       177.000000
mean      42764.672429
std       54411.683611
min           0.040000
25%        5771.180000
50%       25548.840000
75%       58417.350000
max      302722.300000
Name: TotalAmount, dtype: float64


In [5]:
df.to_csv("cleaned_sales_dataset.csv", index=False)

from google.colab import files
files.download("cleaned_sales_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Data Cleaning Summary

The sales dataset was cleaned using Python Pandas in Google Colab.

The following steps were performed:

1. Removed duplicate records.
2. Handled missing values using suitable methods.
3. Corrected invalid and outlier values in Quantity, UnitPrice, and CustomerAge.
4. Converted columns into appropriate data types.
5. Standardized city, product, and order status text.
6. Removed extra spaces.
7. Converted dates into a standard date format and removed invalid dates.
8. Created a new TotalAmount column using Quantity × UnitPrice.
9. Exported the cleaned dataset as cleaned_sales_dataset.csv.

The cleaned dataset is ready for further analysis.
